In [ ]:
%%writefile clean_track_b_embedding_ensemble.py

# Final English ensemble:
#   G2 seed 42 embeddings + Qwen3 non-instruction PC1 embeddings
#   weights = 0.90, 0.10

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score


def norm_text(text):
    return " ".join(str(text).split())


def l2_normalize(matrix):
    matrix = matrix.astype(np.float32, copy=False)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)


def parse_paths(values):
    paths = []
    for value in values:
        for part in str(value).split(","):
            part = part.strip()
            if part:
                paths.append(Path(part))
    return paths


def parse_names(raw_names, paths):
    if raw_names is None:
        return [path.stem for path in paths]
    names = [part.strip() for part in raw_names.split(",") if part.strip()]
    if len(names) != len(paths):
        raise ValueError(f"Expected {len(paths)} names, got {len(names)}")
    return names


def parse_weights(raw_weights, n):
    if raw_weights is None:
        return np.ones(n, dtype=np.float32) / float(n)
    weights = np.asarray(
        [float(part.strip()) for part in raw_weights.split(",") if part.strip()],
        dtype=np.float32,
    )
    if len(weights) != n:
        raise ValueError(f"Expected {n} weights, got {len(weights)}")
    if np.any(weights < 0):
        raise ValueError("Weights must be non-negative")
    total = float(weights.sum())
    if total <= 0:
        raise ValueError("Weights must sum to a positive value")
    return weights / total


def macro_f1_from_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    tp = int((y_pred & y_true).sum())
    tn = int((~y_pred & ~y_true).sum())
    fp = int((y_pred & ~y_true).sum())
    fn = int((~y_pred & y_true).sum())
    f1_pos = 2 * tp / (2 * tp + fp + fn + 1e-9)
    f1_neg = 2 * tn / (2 * tn + fn + fp + 1e-9)
    return (f1_pos + f1_neg) / 2.0, tp, tn, fp, fn


def load_translation_cache(path):
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open(encoding="utf-8") as handle:
        raw = json.load(handle)
    if not isinstance(raw, dict):
        raise ValueError("Translation cache must be a JSON object mapping English text to translated text")
    return {norm_text(k): norm_text(v) for k, v in raw.items()}


def load_embedding_matrix(path, expected_rows=None):
    if not path.exists():
        raise FileNotFoundError(path)
    matrix = np.load(path, allow_pickle=False)
    if matrix.ndim != 2:
        raise ValueError(f"{path} must be a 2D matrix, got shape={matrix.shape}")
    if expected_rows is not None and matrix.shape[0] != expected_rows:
        raise ValueError(f"{path} has {matrix.shape[0]} rows, expected {expected_rows}")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{path} contains NaN or Inf values")
    return l2_normalize(matrix.astype(np.float32))


def load_all_embeddings(paths):
    matrices = []
    expected_rows = None
    for path in paths:
        matrix = load_embedding_matrix(path, expected_rows=expected_rows)
        expected_rows = matrix.shape[0]
        matrices.append(matrix)
    return matrices


def ensemble_concat(matrices, weights):
    blocks = [
        float(np.sqrt(weight)) * matrix
        for matrix, weight in zip(matrices, weights)
    ]
    return l2_normalize(np.concatenate(blocks, axis=1).astype(np.float32))


def ensemble_average(matrices, weights):
    dims = {matrix.shape[1] for matrix in matrices}
    if len(dims) != 1:
        raise ValueError("Average mode requires all matrices to have the same embedding dimension")
    merged = sum(float(weight) * matrix for matrix, weight in zip(matrices, weights))
    return l2_normalize(merged.astype(np.float32))


def ensemble_embeddings(matrices, weights, mode):
    if mode == "concat":
        return ensemble_concat(matrices, weights)
    if mode == "average":
        return ensemble_average(matrices, weights)
    raise ValueError(f"Unsupported mode: {mode}")


def validate_track_b_rows(track_b_input_path, embeddings):
    if track_b_input_path is None:
        return None
    df = pd.read_json(track_b_input_path, lines=True)
    if "text" not in df.columns:
        raise ValueError(f"{track_b_input_path} must contain a 'text' column")
    if len(df) != embeddings.shape[0]:
        raise ValueError(
            f"Track B input rows={len(df)} but embedding rows={embeddings.shape[0]}"
        )
    return df


def map_label_texts(labels_df, translation_cache):
    labels_df = labels_df.copy()
    missing_translation_fields = 0

    for col in ["anchor_text", "text_a", "text_b"]:
        mapped_values = []
        for value in labels_df[col].astype(str).tolist():
            key = norm_text(value)
            if translation_cache is None:
                mapped_values.append(key)
                continue
            mapped = translation_cache.get(key)
            if mapped is None:
                missing_translation_fields += 1
                mapped = key
            mapped_values.append(mapped)
        labels_df[col] = mapped_values

    return labels_df, missing_translation_fields


def evaluate_track_b(track_b_input_path, labels_path, embeddings, translation_cache=None, predictions_out=None):
    track_b_df = validate_track_b_rows(track_b_input_path, embeddings)
    lookup = dict(zip(track_b_df["text"].astype(str).map(norm_text), embeddings))

    labels_df = pd.read_json(labels_path, lines=True)
    required = {"anchor_text", "text_a", "text_b", "text_a_is_closer"}
    missing_cols = sorted(required - set(labels_df.columns))
    if missing_cols:
        raise ValueError(f"{labels_path} is missing required columns: {missing_cols}")

    labels_df, missing_translation_fields = map_label_texts(labels_df, translation_cache)
    labels_df["anchor_embedding"] = labels_df["anchor_text"].map(lookup)
    labels_df["a_embedding"] = labels_df["text_a"].map(lookup)
    labels_df["b_embedding"] = labels_df["text_b"].map(lookup)

    missing_embedding_rows = int(
        (
            labels_df["anchor_embedding"].isna()
            | labels_df["a_embedding"].isna()
            | labels_df["b_embedding"].isna()
        ).sum()
    )
    if missing_embedding_rows:
        labels_df = labels_df[
            ~(
                labels_df["anchor_embedding"].isna()
                | labels_df["a_embedding"].isna()
                | labels_df["b_embedding"].isna()
            )
        ].copy()

    y_true = labels_df["text_a_is_closer"].astype(bool).tolist()
    margins = []
    y_pred = []
    for _, row in labels_df.iterrows():
        score_a = float(row["anchor_embedding"].dot(row["a_embedding"]))
        score_b = float(row["anchor_embedding"].dot(row["b_embedding"]))
        margin = score_a - score_b
        margins.append(margin)
        y_pred.append(margin > 0)

    acc = accuracy_score(y_true, y_pred)
    macro_f1, tp, tn, fp, fn = macro_f1_from_binary(y_true, y_pred)

    if predictions_out is not None:
        out_path = Path(predictions_out)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with out_path.open("w", encoding="utf-8") as handle:
            for index, (gold, pred, margin) in enumerate(zip(y_true, y_pred, margins)):
                payload = {
                    "row_id": index,
                    "gold_text_a_is_closer": bool(gold),
                    "pred_text_a_is_closer": bool(pred),
                    "margin": float(margin),
                    "correct": bool(gold == pred),
                }
                handle.write(json.dumps(payload) + "\n")

    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "n": int(len(y_true)),
        "missing_translation_fields": int(missing_translation_fields),
        "missing_embedding_rows": int(missing_embedding_rows),
        "embedding_dim": int(embeddings.shape[1]),
        "mean_abs_margin": float(np.mean(np.abs(margins))) if margins else 0.0,
    }


def make_diagnostics(names, paths, matrices, weights, mode, out_path, ensemble):
    return {
        "mode": mode,
        "components": [
            {
                "name": name,
                "path": str(path),
                "shape": list(matrix.shape),
                "weight": float(weight),
            }
            for name, path, matrix, weight in zip(names, paths, matrices, weights)
        ],
        "output": str(out_path),
        "output_shape": list(ensemble.shape),
        "row_norm_mean": float(np.linalg.norm(ensemble, axis=1).mean()),
        "row_norm_min": float(np.linalg.norm(ensemble, axis=1).min()),
        "row_norm_max": float(np.linalg.norm(ensemble, axis=1).max()),
    }


def main():
    parser = argparse.ArgumentParser(
        description="Clean weighted ensemble for Track B embedding matrices."
    )
    parser.add_argument(
        "--embeddings",
        nargs="+",
        required=True,
        help="Input .npy embedding files. Space-separated or comma-separated paths are accepted.",
    )
    parser.add_argument(
        "--names",
        default=None,
        help="Optional comma-separated component names, e.g. g2,qwen3_pc1.",
    )
    parser.add_argument(
        "--weights",
        default=None,
        help="Optional comma-separated weights. If omitted, equal weights are used.",
    )
    parser.add_argument(
        "--mode",
        choices=["concat", "average"],
        default="concat",
        help="concat supports different dimensions; average requires equal dimensions.",
    )
    parser.add_argument("--out", required=True, help="Output .npy ensemble path.")
    parser.add_argument("--diagnostics", default=None, help="Optional diagnostics JSON path.")
    parser.add_argument("--track-b-input", default=None, help="Optional Track B JSONL story file.")
    parser.add_argument("--labels", default=None, help="Optional Track B labels JSONL file.")
    parser.add_argument(
        "--translation-cache",
        default=None,
        help="Optional English-to-target-language JSON cache for Romanian Track B evaluation.",
    )
    parser.add_argument(
        "--predictions-out",
        default=None,
        help="Optional JSONL file with per-label-row predictions and margins.",
    )
    args = parser.parse_args()

    paths = parse_paths(args.embeddings)
    if len(paths) < 2:
        raise ValueError("Provide at least two embedding files.")

    names = parse_names(args.names, paths)
    weights = parse_weights(args.weights, len(paths))
    matrices = load_all_embeddings(paths)
    ensemble = ensemble_embeddings(matrices, weights, args.mode)

    if args.track_b_input is not None:
        validate_track_b_rows(args.track_b_input, ensemble)

    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(out_path, ensemble.astype(np.float32))

    diagnostics = make_diagnostics(names, paths, matrices, weights, args.mode, out_path, ensemble)

    if args.labels is not None:
        if args.track_b_input is None:
            raise ValueError("--track-b-input is required when --labels is provided")
        translation_cache = load_translation_cache(args.translation_cache)
        diagnostics["track_b_eval"] = evaluate_track_b(
            args.track_b_input,
            args.labels,
            ensemble,
            translation_cache=translation_cache,
            predictions_out=args.predictions_out,
        )
        diagnostics["translation_cache"] = args.translation_cache
        diagnostics["predictions_out"] = args.predictions_out

    diagnostics_path = Path(args.diagnostics) if args.diagnostics else out_path.with_suffix(".diagnostics.json")
    diagnostics_path.parent.mkdir(parents=True, exist_ok=True)
    diagnostics_path.write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")

    print(f"Saved ensemble embeddings: {out_path} shape={ensemble.shape}")
    print(f"Saved diagnostics: {diagnostics_path}")
    if "track_b_eval" in diagnostics:
        metrics = diagnostics["track_b_eval"]
        print(f"Track B accuracy : {metrics['accuracy'] * 100:.2f}%")
        print(f"Track B macro-F1 : {metrics['macro_f1'] * 100:.2f}%")
        print(f"Mean abs margin  : {metrics['mean_abs_margin']:.6f}")


if __name__ == "__main__":
    main()

Writing clean_track_b_embedding_ensemble.py


In [38]:
!python clean_track_b_embedding_ensemble.py \
  --embeddings \
    /kaggle/input/datasets/anisoaraana/similarity/G2_512_full_train_track_b.npy \
    /kaggle/input/datasets/anisoaraana/similarity/qwen3_06b_zero_shot_track_b.npy \
  --names g2_seed42,qwen3_pc1 \
  --weights 0.90,0.10 \
  --mode concat \
  --out /kaggle/working/g2_qwen3_pc1_ensemble_track_b.npy \
  --track-b-input /kaggle/input/datasets/anisoaraana/similarity/test_track_b.jsonl \
  --labels /kaggle/input/datasets/anisoaraana/similarity/test_track_b_labels.jsonl \
  --predictions-out /kaggle/working/g2_qwen3_pc1_ensemble_predictions.jsonl


Saved ensemble embeddings: /kaggle/working/g2_qwen3_pc1_ensemble_track_b.npy shape=(849, 2560)
Saved diagnostics: /kaggle/working/g2_qwen3_pc1_ensemble_track_b.diagnostics.json
Track B accuracy : 72.00%
Track B macro-F1 : 71.98%
Mean abs margin  : 0.121852
